In [1]:
DOMAIN = "facilities_and_campus_services"
COLLECTION_NAME = "facilities_and_campus_services"

In [2]:
from pathlib import Path
import io
import re

import fitz
from PIL import Image
import pytesseract

import chromadb
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_chroma import Chroma

c:\Users\shafw\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# =====================
# CONFIG
# =====================

BASE_DIR = Path(r"D:\code\nlp\ta-sisdas")

DOMAIN = "facilities_and_campus_services"
DOMAIN_DIR = BASE_DIR / "dataset_raw" / DOMAIN
CHROMA_DIR = BASE_DIR / "chroma_db"

COLLECTION_NAME = "facilities_and_campus_services"
EMBEDDING_MODEL = "bge-m3"

RESET_COLLECTION = True

pytesseract.pytesseract.tesseract_cmd = r"C:\Program Files\Tesseract-OCR\tesseract.exe"

In [4]:
# =====================
# MANUAL DOCUMENT METADATA
# =====================

DOCUMENT_METADATA_MAP = {
    "Sarana dan Prasarana perpustakaan um.pdf": {
        "document_type": "facility_info",
        "academic_year": "general",
        "document_year": "general",
        "topic": "library_facilities"
    },

    "Sarana Umum um.pdf": {
        "document_type": "facility_info",
        "academic_year": "general",
        "document_year": "general",
        "topic": "general_facilities"
    },

    "SOP-GRAFIS-Tri-Wahyuningtyas.pdf": {
        "document_type": "facility_sop",
        "academic_year": "general",
        "document_year": "general",
        "topic": "graphic_studio"
    },

    "SOP-LAB-SOSIO-UM-Deny-Wahyu-Apriadi.pdf": {
        "document_type": "facility_sop",
        "academic_year": "general",
        "document_year": "general",
        "topic": "sociology_laboratory"
    },

    "SOP-LAB-TM-REF-7.pdf": {
        "document_type": "facility_sop",
        "academic_year": "general",
        "document_year": "general",
        "topic": "laboratory"
    },

    "SOP-MEDIA-REKAM-Tri-Wahyuningtyas.pdf": {
        "document_type": "facility_sop",
        "academic_year": "general",
        "document_year": "general",
        "topic": "recording_media"
    },

    "SOP-STUDIO-LUKIS-Tri-Wahyuningtyas.pdf": {
        "document_type": "facility_sop",
        "academic_year": "general",
        "document_year": "general",
        "topic": "painting_studio"
    },

    "SOP-STUDIO-MUSIK-Tri-Wahyuningtyas.pdf": {
        "document_type": "facility_sop",
        "academic_year": "general",
        "document_year": "general",
        "topic": "music_studio"
    },

    "SOP-STUDIO-TARI-Tri-Wahyuningtyas.pdf": {
        "document_type": "facility_sop",
        "academic_year": "general",
        "document_year": "general",
        "topic": "dance_studio"
    },

    "TARIF-PENGGUNAAN-ALAT-LAB-SOSIO-Deny-Wahyu-Apriadi.pdf": {
        "document_type": "facility_fee_info",
        "academic_year": "general",
        "document_year": "general",
        "topic": "sociology_laboratory_fee"
    }
}


def get_document_metadata(pdf_path: Path) -> dict:
    file_name = pdf_path.name

    if file_name not in DOCUMENT_METADATA_MAP:
        raise ValueError(
            f"Metadata untuk file ini belum diset manual: {file_name}"
        )

    return DOCUMENT_METADATA_MAP[file_name]

In [5]:
# =====================
# HELPER
# =====================

def make_safe_id(text: str) -> str:
    text = text.replace(" ", "_")
    text = re.sub(r"[^a-zA-Z0-9_\-]", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def load_pdf_normal(pdf_path: Path, domain: str):
    loader = PyPDFLoader(str(pdf_path))
    docs = loader.load()

    cleaned_docs = []
    extra_metadata = get_document_metadata(pdf_path)

    for doc in docs:
        text = doc.page_content.strip()

        if not text:
            continue

        page = doc.metadata.get("page", 0)

        try:
            page = int(page) + 1
        except Exception:
            page = None

        doc.metadata = {
            "domain": domain,
            "source": str(pdf_path),
            "file_name": pdf_path.name,
            "page": page,
            "extraction_method": "pypdf",
            **extra_metadata
        }

        cleaned_docs.append(doc)

    return cleaned_docs


def load_pdf_ocr(pdf_path: Path, domain: str):
    pdf = fitz.open(str(pdf_path))
    docs = []
    extra_metadata = get_document_metadata(pdf_path)

    for page_number, page in enumerate(pdf, start=1):
        pix = page.get_pixmap(matrix=fitz.Matrix(3, 3), alpha=False)
        image = Image.open(io.BytesIO(pix.tobytes("png")))

        text = pytesseract.image_to_string(image, lang="ind+eng").strip()

        print(f"  OCR page {page_number}: {len(text)} chars")

        if text:
            docs.append(
                Document(
                    page_content=text,
                    metadata={
                        "domain": domain,
                        "source": str(pdf_path),
                        "file_name": pdf_path.name,
                        "page": page_number,
                        "extraction_method": "ocr_tesseract_ind_eng",
                        **extra_metadata
                    }
                )
            )

    pdf.close()
    return docs


def load_pdf_smart(pdf_path: Path, domain: str, min_chars: int = 100):
    print(f"\nLoading: {pdf_path.name}")

    normal_docs = load_pdf_normal(pdf_path, domain)
    normal_chars = sum(len(doc.page_content) for doc in normal_docs)

    print(f"  Normal extraction chars: {normal_chars}")

    if normal_chars >= min_chars:
        print("  Using normal extraction")
        return normal_docs

    print("  Normal extraction too small. Using OCR")
    return load_pdf_ocr(pdf_path, domain)

In [6]:
# =====================
# PREPARE CHROMA
# =====================

embeddings = OllamaEmbeddings(
    model=EMBEDDING_MODEL
)

test_vector = embeddings.embed_query("tes embedding")
print("Embedding dimension:", len(test_vector))

if len(test_vector) == 0:
    raise ValueError("Embedding gagal. Pastikan Ollama jalan dan model bge-m3 sudah di-pull.")


client = chromadb.PersistentClient(path=str(CHROMA_DIR))

if RESET_COLLECTION:
    try:
        client.delete_collection(name=COLLECTION_NAME)
        print(f"Old collection deleted: {COLLECTION_NAME}")
    except Exception:
        print(f"No old collection found: {COLLECTION_NAME}")


vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(CHROMA_DIR)
)

Embedding dimension: 1024
No old collection found: facilities_and_campus_services


In [7]:
# =====================
# TEXT SPLITTER
# =====================

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=500,
    chunk_overlap=100
)

In [8]:
# =====================
# INDEX ALL PDF IN DOMAIN
# =====================

pdf_files = sorted(DOMAIN_DIR.glob("*.pdf"))

print(f"\nTotal PDF files found: {len(pdf_files)}")

total_docs = 0
total_chunks = 0
failed_files = []

for pdf_path in pdf_files:
    try:
        docs = load_pdf_smart(pdf_path, DOMAIN)

        print(f"  Total docs/pages loaded: {len(docs)}")

        if len(docs) == 0:
            print("  Skipped: no text extracted")
            failed_files.append((pdf_path.name, "No text extracted"))
            continue

        splits = text_splitter.split_documents(docs)

        splits = [
            split for split in splits
            if split.page_content and split.page_content.strip()
        ]

        file_key = make_safe_id(pdf_path.stem)

        for i, split in enumerate(splits):
            split.metadata["chunk_index"] = i
            split.metadata["file_key"] = file_key

        ids = [
            f"{DOMAIN}-{file_key}-page-{doc.metadata.get('page')}-chunk-{doc.metadata.get('chunk_index')}"
            for doc in splits
        ]

        print(f"  Total chunks: {len(splits)}")

        if len(splits) == 0:
            print("  Skipped: no chunks created")
            failed_files.append((pdf_path.name, "No chunks created"))
            continue

        vectorstore.add_documents(
            documents=splits,
            ids=ids
        )

        total_docs += len(docs)
        total_chunks += len(splits)

        print(f"  Indexed successfully: {pdf_path.name}")

    except Exception as e:
        print(f"  Failed: {pdf_path.name}")
        print(f"  Error: {e}")
        failed_files.append((pdf_path.name, str(e)))


Total PDF files found: 10

Loading: Sarana dan Prasarana perpustakaan um.pdf
  Normal extraction chars: 1284
  Using normal extraction
  Total docs/pages loaded: 1
  Total chunks: 2
  Indexed successfully: Sarana dan Prasarana perpustakaan um.pdf

Loading: Sarana Umum um.pdf
  Normal extraction chars: 23789
  Using normal extraction
  Total docs/pages loaded: 9
  Total chunks: 26
  Indexed successfully: Sarana Umum um.pdf

Loading: SOP-GRAFIS-Tri-Wahyuningtyas.pdf
  Normal extraction chars: 5923
  Using normal extraction
  Total docs/pages loaded: 3
  Total chunks: 7
  Indexed successfully: SOP-GRAFIS-Tri-Wahyuningtyas.pdf

Loading: SOP-LAB-SOSIO-UM-Deny-Wahyu-Apriadi.pdf
  Normal extraction chars: 16547
  Using normal extraction
  Total docs/pages loaded: 18
  Total chunks: 24
  Indexed successfully: SOP-LAB-SOSIO-UM-Deny-Wahyu-Apriadi.pdf

Loading: SOP-LAB-TM-REF-7.pdf
  Normal extraction chars: 4657
  Using normal extraction
  Total docs/pages loaded: 8
  Total chunks: 8
  Indexed 

In [9]:
# =====================
# SUMMARY
# =====================

print("\n" + "=" * 100)
print("INDEXING SUMMARY")
print("=" * 100)

print("Domain:", DOMAIN)
print("Collection:", COLLECTION_NAME)
print("Total PDFs:", len(pdf_files))
print("Total docs/pages:", total_docs)
print("Total chunks indexed:", total_chunks)
print("Total data in collection:", vectorstore._collection.count())

if failed_files:
    print("\nFailed files:")
    for file_name, reason in failed_files:
        print("-", file_name, "=>", reason)
else:
    print("\nNo failed files.")


INDEXING SUMMARY
Domain: facilities_and_campus_services
Collection: facilities_and_campus_services
Total PDFs: 10
Total docs/pages: 53
Total chunks indexed: 96
Total data in collection: 96

No failed files.


In [10]:
from collections import defaultdict

collection = client.get_collection(name=COLLECTION_NAME)

data = collection.get(include=["metadatas"])

metadata_values = defaultdict(set)

for metadata in data["metadatas"]:
    for key, value in metadata.items():
        metadata_values[key].add(value)

for key, values in metadata_values.items():
    print("=" * 80)
    print("Metadata key:", key)
    print("Values:")
    for value in sorted(values, key=lambda x: str(x)):
        print("-", value)

Metadata key: domain
Values:
- facilities_and_campus_services
Metadata key: topic
Values:
- dance_studio
- general_facilities
- graphic_studio
- laboratory
- library_facilities
- music_studio
- painting_studio
- recording_media
- sociology_laboratory
- sociology_laboratory_fee
Metadata key: academic_year
Values:
- general
Metadata key: file_key
Values:
- SOP-GRAFIS-Tri-Wahyuningtyas
- SOP-LAB-SOSIO-UM-Deny-Wahyu-Apriadi
- SOP-LAB-TM-REF-7
- SOP-MEDIA-REKAM-Tri-Wahyuningtyas
- SOP-STUDIO-LUKIS-Tri-Wahyuningtyas
- SOP-STUDIO-MUSIK-Tri-Wahyuningtyas
- SOP-STUDIO-TARI-Tri-Wahyuningtyas
- Sarana_Umum_um
- Sarana_dan_Prasarana_perpustakaan_um
- TARIF-PENGGUNAAN-ALAT-LAB-SOSIO-Deny-Wahyu-Apriadi
Metadata key: file_name
Values:
- SOP-GRAFIS-Tri-Wahyuningtyas.pdf
- SOP-LAB-SOSIO-UM-Deny-Wahyu-Apriadi.pdf
- SOP-LAB-TM-REF-7.pdf
- SOP-MEDIA-REKAM-Tri-Wahyuningtyas.pdf
- SOP-STUDIO-LUKIS-Tri-Wahyuningtyas.pdf
- SOP-STUDIO-MUSIK-Tri-Wahyuningtyas.pdf
- SOP-STUDIO-TARI-Tri-Wahyuningtyas.pdf
- Sarana

In [11]:
from collections import Counter

data = collection.get(include=["metadatas"])

counter = Counter(
    metadata.get("file_name", "UNKNOWN")
    for metadata in data["metadatas"]
)

for file_name, count in counter.items():
    print(file_name, ":", count, "chunks")

Sarana dan Prasarana perpustakaan um.pdf : 2 chunks
Sarana Umum um.pdf : 26 chunks
SOP-GRAFIS-Tri-Wahyuningtyas.pdf : 7 chunks
SOP-LAB-SOSIO-UM-Deny-Wahyu-Apriadi.pdf : 24 chunks
SOP-LAB-TM-REF-7.pdf : 8 chunks
SOP-MEDIA-REKAM-Tri-Wahyuningtyas.pdf : 7 chunks
SOP-STUDIO-LUKIS-Tri-Wahyuningtyas.pdf : 6 chunks
SOP-STUDIO-MUSIK-Tri-Wahyuningtyas.pdf : 7 chunks
SOP-STUDIO-TARI-Tri-Wahyuningtyas.pdf : 7 chunks
TARIF-PENGGUNAAN-ALAT-LAB-SOSIO-Deny-Wahyu-Apriadi.pdf : 2 chunks
